In [4]:
import csv
from pathlib import Path
import polars as pl
import chardet

listing_file_path = "Listings.csv"

In [ ]:
with open(listing_file_path ,"rb") as f:
    raw = f.read()

chardet.detect(raw)

{'encoding': 'utf-8',
 'confidence': 0.8034029,
 'language': 'en',
 'mime_type': 'text/plain'}

In [ ]:
def safe_decode(value: bytes):
    try:
        decoded = value.decode("utf-8")
        return decoded, False
    except UnicodeDecodeError:
        decoded = value.decode("utf-8", errors="replace")
        return decoded, True


rows = []

with open(listing_file_path, "rb") as f:
    reader = csv.reader(
        (line.decode("latin1") for line in f)
    )

    headers = next(reader)

    for row_idx, row in enumerate(reader):
        parsed_row = {}
        row_has_non_utf8 = False

        for col_name, value in zip(headers, row):
            raw_bytes = value.encode("latin1")
            decoded_value, has_non_utf8 = safe_decode(raw_bytes)
            parsed_row[col_name] = decoded_value
            parsed_row[f"{col_name}_non_utf8"] = has_non_utf8

            if has_non_utf8:
                row_has_non_utf8 = True

        parsed_row["non_utf8"] = row_has_non_utf8
        rows.append(parsed_row)


df = pl.DataFrame(rows)

df.head()

listing_id,listing_id_non_utf8,name,name_non_utf8,host_id,host_id_non_utf8,host_since,host_since_non_utf8,host_location,host_location_non_utf8,host_response_time,host_response_time_non_utf8,host_response_rate,host_response_rate_non_utf8,host_acceptance_rate,host_acceptance_rate_non_utf8,host_is_superhost,host_is_superhost_non_utf8,host_total_listings_count,host_total_listings_count_non_utf8,host_has_profile_pic,host_has_profile_pic_non_utf8,host_identity_verified,host_identity_verified_non_utf8,neighbourhood,neighbourhood_non_utf8,district,district_non_utf8,city,city_non_utf8,latitude,latitude_non_utf8,longitude,longitude_non_utf8,property_type,property_type_non_utf8,room_type,room_type_non_utf8,accommodates,accommodates_non_utf8,bedrooms,bedrooms_non_utf8,amenities,amenities_non_utf8,price,price_non_utf8,minimum_nights,minimum_nights_non_utf8,maximum_nights,maximum_nights_non_utf8,review_scores_rating,review_scores_rating_non_utf8,review_scores_accuracy,review_scores_accuracy_non_utf8,review_scores_cleanliness,review_scores_cleanliness_non_utf8,review_scores_checkin,review_scores_checkin_non_utf8,review_scores_communication,review_scores_communication_non_utf8,review_scores_location,review_scores_location_non_utf8,review_scores_value,review_scores_value_non_utf8,instant_bookable,instant_bookable_non_utf8,non_utf8
str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,str,bool,bool
"""281420""",false,"""Beautiful Flat in le Village M…",false,"""1466919""",false,"""2011-12-03""",false,"""Paris, Ile-de-France, France""",false,"""""",false,"""""",false,"""""",false,"""f""",false,"""1""",false,"""t""",false,"""f""",false,"""Buttes-Montmartre""",false,"""""",false,"""Paris""",false,"""48.88668""",false,"""2.33343""",false,"""Entire apartment""",false,"""Entire place""",false,"""2""",false,"""1""",false,"""[""Heating"", ""Kitchen"", ""Washer…",false,"""53""",false,"""2""",false,"""1125""",false,"""100""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""f""",false,false
"""3705183""",false,"""39 mÂ² Paris (Sacre CÅ“ur)""",false,"""10328771""",false,"""2013-11-29""",false,"""Paris, Ile-de-France, France""",false,"""""",false,"""""",false,"""""",false,"""f""",false,"""1""",false,"""t""",false,"""t""",false,"""Buttes-Montmartre""",false,"""""",false,"""Paris""",false,"""48.88617""",false,"""2.34515""",false,"""Entire apartment""",false,"""Entire place""",false,"""2""",false,"""1""",false,"""[""Shampoo"", ""Heating"", ""Kitche…",false,"""120""",false,"""2""",false,"""1125""",false,"""100""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""f""",false,false
"""4082273""",false,"""Lovely apartment with Terrace,…",false,"""19252768""",false,"""2014-07-31""",false,"""Paris, Ile-de-France, France""",false,"""""",false,"""""",false,"""""",false,"""f""",false,"""1""",false,"""t""",false,"""f""",false,"""Elysee""",false,"""""",false,"""Paris""",false,"""48.88112""",false,"""2.31712""",false,"""Entire apartment""",false,"""Entire place""",false,"""2""",false,"""1""",false,"""[""Heating"", ""TV"", ""Kitchen"", ""…",false,"""89""",false,"""2""",false,"""1125""",false,"""100""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""10""",false,"""f""",false,false
"""4797344""",false,"""Cosy studio (close to Eiffel t…",false,"""10668311""",false,"""2013-12-17""",false,"""Paris, Ile-de-France, France""",false,"""""",false,"""""",false,"""""",false,"""f""",false,"""1""",false,"""t""",false,"""t""",false,"""Vaugirard""",false,"""""",false,"""Paris""",false,"""48.84571""",false,"""2.30584""",false,"""Entire apartment""",false,"""Entire place""",false,"""2""",false,"""1""",false,"""[""Heating"", ""TV"

In [3]:
df.shape

(279712, 67)

In [2]:
import polars as pl


non_utf8_cols = [
    col for col in df.columns
    if col.endswith("_non_utf8")
]

result = {}

for flag_col in non_utf8_cols:

    original_col = flag_col.replace("_non_utf8", "")

    unique_values = (
        df.filter(pl.col(flag_col))
        .select(original_col)
        .unique()
        .to_series()
        .to_list()
    )
    if unique_values:
        result[original_col] = unique_values


result

{'name': ['A�lvaro Obregon 272',
  'Newly Renovated apt in front of A�lvaro Obregon!!',
  'Amplia habitacion en Av. A�lvaro Obregon & Frontera',
  'A�lvaro Obregon/ Roma cozy Great location 3',
  'Departamento completo en A�lvaro Obregon.'],
 'host_location': ['A�lvaro Obregon, Mexico City, Mexico',
  'A�lvaro Obregon, Federal District, Mexico'],
 'neighbourhood': ['A�lvaro Obregon']}

In [1]:
import locale
locale.getpreferredencoding()

'UTF-8'

In [ ]:
# read with utf 8 -> error -> detect non utf8 -> fix non utf 8 -> read again
#  Source: https://unix.stackexchange.com/questions/6516/filtering-invalid-utf8
#  grep -axv '.*' /home/user/dapractice/lesson/16th/Listings.csv > invalid.txt => write all lines contain atleast 1 non utf8 chars
#  using ftyfty to fix non utf8 chars: ftyfty.fix_text()


from charset_normalizer import from_bytes
from pathlib import Path

raw = Path("/home/user/dapractice/lesson/16th/Listings.csv").read_bytes()

result = from_bytes(raw).best()

print(result.encoding)

text = str(result)

Path("listings_charset_normalizer_fixed.csv").write_text(text, encoding="utf-8")

hp_roman8


158497169

In [11]:
import polars as pl

df_listing: pl.DataFrame = (
    pl.read_csv(
        "listings_charset_normalizer_fixed.csv",
        has_header=True,
        infer_schema_length=0,
        truncate_ragged_lines=True,
        encoding="utf8"
    )
)
df_listing.head()

listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood,district,city,latitude,longitude,property_type,room_type,accommodates,bedrooms,amenities,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""281420""","""Beautiful Flat in le Village M…","""1466919""","""2011-12-03""","""Paris, Ile-de-France, France""",null,null,null,"""f""","""1""","""t""","""f""","""Buttes-Montmartre""",null,"""Paris""","""48.88668""","""2.33343""","""Entire apartment""","""Entire place""","""2""","""1""","""[""Heating"", ""Kitchen"", ""Washer…","""53""","""2""","""1125""","""100""","""10""","""10""","""10""","""10""","""10""","""10""","""f"""
"""3705183""","""39 mûôý Paris (Sacre Cû ãur…","""10328771""","""2013-11-29""","""Paris, Ile-de-France, France""",null,null,null,"""f""","""1""","""t""","""t""","""Buttes-Montmartre""",null,"""Paris""","""48.88617""","""2.34515""","""Entire apartment""","""Entire place""","""2""","""1""","""[""Shampoo"", ""Heating"", ""Kitche…","""120""","""2""","""1125""","""100""","""10""","""10""","""10""","""10""","""10""","""10""","""f"""
"""4082273""","""Lovely apartment with Terrace,…","""19252768""","""2014-07-31""","""Paris, Ile-de-France, France""",null,null,null,"""f""","""1""","""t""","""f""","""Elysee""",null,"""Paris""","""48.88112""","""2.31712""","""Entire apartment""","""Entire place""","""2""","""1""","""[""Heating"", ""TV"", ""Kitchen"", ""…","""89""","""2""","""1125""","""100""","""10""","""10""","""10""","""10""","""10""","""10""","""f"""
"""4797344""","""Cosy studio (close to Eiffel t…","""10668311""","""2013-12-17""","""Paris, Ile-de-France, France""",null,null,null,"""f""","""1""","""t""","""t""","""Vaugirard""",null,"""Paris""","""48.84571""","""2.30584""","""Entire apartment""","""Entire place""","""2""","""1""","""[""Heating"", ""TV"", ""Kitchen"", ""…","""58""","""2""","""1125""","""100""","""10""","""10""","""10""","""10""","""10""","""10""","""f"""
"""4823489""","""Close to Eiffel Tower - Beauti…","""24837558""","""2014-12-14""","""Paris, Ile-de-France, France""",null,null,null,"""f""","""1""","""t""","""f""","""Passy""",null,"""Paris""","""48.855""","""2.26979""","""Entire apartment""","""Entire place""","""2""","""1""","""[""Heating"", ""TV"", ""Kitchen"", ""…","""60""","""2""","""1125""","""100""","""10""","""10""","""10""","""10""","""10""","""10""","""f"""


In [14]:
def detect_non_utf8(filepath: str) -> list[dict]:
    """Returns a list of problem locations with line, column, and raw bytes."""
    issues = []
    with open(filepath, "rb") as f:  # open as raw bytes
        for line_num, line in enumerate(f, start=1):
            try:
                line.decode("utf-8")
            except UnicodeDecodeError as e:
                issues.append({
                    "line": line_num,
                    "start": e.start,
                    "end": e.end,
                    "bad_bytes": line[e.start:e.end].hex(),
                    "reason": e.reason,
                })
    return issues

issues = detect_non_utf8("/home/user/dapractice/lesson/16th/Listings.csv")
for i in issues:
    print(f"Line {i['line']}, col {i['start']}: bad bytes 0x{i['bad_bytes']} ({i['reason']})")

Line 11877, col 110: bad bytes 0x81 (invalid start byte)
Line 11942, col 132: bad bytes 0x81 (invalid start byte)
Line 12540, col 127: bad bytes 0x81 (invalid start byte)
Line 12581, col 96: bad bytes 0x81 (invalid start byte)
Line 13606, col 104: bad bytes 0x81 (invalid start byte)
Line 13908, col 104: bad bytes 0x81 (invalid start byte)
Line 14361, col 78: bad bytes 0x81 (invalid start byte)
Line 14594, col 126: bad bytes 0x81 (invalid start byte)
Line 17151, col 115: bad bytes 0x81 (invalid start byte)
Line 17754, col 98: bad bytes 0x81 (invalid start byte)
Line 17903, col 120: bad bytes 0x81 (invalid start byte)
Line 19423, col 113: bad bytes 0x81 (invalid start byte)
Line 21344, col 113: bad bytes 0x81 (invalid start byte)
Line 22524, col 107: bad bytes 0x81 (invalid start byte)
Line 22730, col 128: bad bytes 0x81 (invalid start byte)
Line 23704, col 70: bad bytes 0x81 (invalid start byte)
Line 26349, col 112: bad bytes 0x81 (invalid start byte)
Line 26422, col 99: bad bytes 0x81 

In [15]:
import chardet

with open("/home/user/dapractice/lesson/16th/Listings.csv", "rb") as f:
    raw = f.read()

result = chardet.detect(raw)
print(result)  # e.g. {'encoding': 'windows-1252', 'confidence': 0.99}

{'encoding': 'utf-8', 'confidence': 0.8034029, 'language': 'en', 'mime_type': 'text/plain'}


In [17]:
def reencode_file(src: str, dest: str, from_encoding="latin-1"):
    with open(src, "r", encoding=from_encoding) as f:
        content = f.read()
    with open(dest, "w", encoding="utf-8") as f:
        f.write(content)

reencode_file("/home/user/dapractice/lesson/16th/Listings.csv", "/home/user/dapractice/lesson/16th/Listings_utf8_01.csv")

In [18]:
def fix_with_errors(src: str, dest: str, errors="replace"):
    """
    errors options:
      'replace'  → bad bytes become the replacement char (U+FFFD: '?')
      'ignore'   → bad bytes are silently dropped
      'xmlcharrefreplace' → bad bytes become &#NNN; HTML entities
    """
    with open(src, "rb") as f:
        raw = f.read()
    fixed = raw.decode("utf-8", errors=errors)
    with open(dest, "w", encoding="utf-8") as f:
        f.write(fixed)

fix_with_errors("/home/user/dapractice/lesson/16th/Listings.csv", "/home/user/dapractice/lesson/16th/Listings_utf8_02.csv", errors="replace")

In [19]:
def strip_non_utf8(src: str, dest: str):
    with open(src, "rb") as f:
        raw = f.read()
    # Encode back to bytes, dropping anything that can't round-trip
    cleaned = raw.decode("utf-8", errors="ignore").encode("utf-8")
    with open(dest, "wb") as f:
        f.write(cleaned)
strip_non_utf8("/home/user/dapractice/lesson/16th/Listings.csv", "/home/user/dapractice/lesson/16th/Listings_utf8_03.csv")

In [20]:
import pandas as pd

df = pd.read_csv(
    "/home/user/dapractice/lesson/16th/Listings.csv",
    encoding="latin-1",       # or use chardet result
    encoding_errors="replace"  # pandas 1.3+
)

df.to_csv("/home/user/dapractice/lesson/16th/Listings_utf8_04.csv", index=False, encoding="utf-8")

/tmp/ipykernel_308/3654663980.py:3: DtypeWarning: Columns (0: host_response_time, 1: district) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [21]:
import chardet
from pathlib import Path

def fix_encoding(src: str, dest: str | None = None, strategy: str = "replace"):
    """
    Detect encoding, then convert to UTF-8.
    strategy: 'replace' | 'ignore' | 'reencode'
    """
    src_path = Path(src)
    dest_path = Path(dest) if dest else src_path.with_stem(src_path.stem + "_utf8")

    raw = src_path.read_bytes()
    detected = chardet.detect(raw)
    enc = detected.get("encoding") or "latin-1"
    conf = detected.get("confidence", 0)

    print(f"Detected: {enc} (confidence: {conf:.0%})")

    if strategy == "reencode":
        text = raw.decode(enc, errors="replace")
    else:
        text = raw.decode("utf-8", errors=strategy)

    dest_path.write_text(text, encoding="utf-8")
    print(f"Saved clean file → {dest_path}")

# Usage
fix_encoding("/home/user/dapractice/lesson/16th/Listings.csv", "/home/user/dapractice/lesson/16th/Listings_utf8_05.csv", strategy="reencode")

Detected: utf-8 (confidence: 80%)
Saved clean file → /home/user/dapractice/lesson/16th/Listings_utf8_05.csv
